# Task 1 — Rakuten Product Classification: EDA & Preprocessing

## 1. Project Introduction

### Context
This notebook addresses the **Rakuten France Multimodal Product Data Classification** challenge
(ENS ChallengeData #35). The task is to automatically assign one of 27 product category codes
(`prdtypecode`) to e-commerce listings using three information sources:

- **`designation`**: short product title (always present)
- **`description`**: longer HTML-rich text (missing in ~35% of rows)
- **Product image**: JPEG photograph (always present)

**Economic context:** Manual product cataloging is a bottleneck at scale.
Accurate automated classification enables better search, recommendation, and inventory management.

**Scientific context:** This is a multimodal classification problem requiring fusion of
NLP (text) and computer vision (image) signals — an active research area.

### Objectives
1. Understand the dataset (volume, quality, class distribution, missing data).
2. Produce ≥5 annotated visualizations with statistical validation.
3. Preprocess text and images so the dataset is ready for Rendu 2 modeling.
4. Save cleaned training and test sets as pickle files.

### Target
Predict `prdtypecode` (27 categories) from `designation` + `description` + product image.
Primary evaluation metric: **weighted F1-score (macro)**.

## 2. Data Understanding

In [1]:
import pandas as pd
import numpy as np
import os
import re
import unicodedata

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.io as pio 
import io
from PIL import Image
from scipy import stats
from scipy.stats import chi2_contingency, kruskal
from sklearn.feature_extraction.text import TfidfVectorizer
from wordcloud import WordCloud
import nltk

nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

import warnings
warnings.filterwarnings('ignore')

# ── Load raw files ──────────────────────────────────────────────────────────
x_train = pd.read_csv('../data/x_train.csv', index_col=0)
y_train = pd.read_csv('../data/y_train.csv', index_col=0)
test_raw = pd.read_csv('../data/test.csv',    index_col=0)

# ── Merge features + labels on shared index ────────────────────────────────
train = x_train.merge(y_train, left_index=True, right_index=True, how='inner')
assert train.shape[0] == x_train.shape[0] == y_train.shape[0], \
    f"Merge row count mismatch: {train.shape[0]} rows (expected {x_train.shape[0]})"

print(f"Training set : {train.shape[0]:,} rows × {train.shape[1]} columns")
print(f"Test set     : {test_raw.shape[0]:,} rows × {test_raw.shape[1]} columns")
print(f"\nTraining columns : {train.columns.tolist()}")
print(f"\nTarget classes   : {sorted(train['prdtypecode'].unique().tolist())}")
print(f"Number of classes: {train['prdtypecode'].nunique()}")

Training set : 84,916 rows × 5 columns
Test set     : 13,812 rows × 4 columns

Training columns : ['designation', 'description', 'productid', 'imageid', 'prdtypecode']

Target classes   : [10, 40, 50, 60, 1140, 1160, 1180, 1280, 1281, 1300, 1301, 1302, 1320, 1560, 1920, 1940, 2060, 2220, 2280, 2403, 2462, 2522, 2582, 2583, 2585, 2705, 2905]
Number of classes: 27


In [2]:
print(x_train.shape, y_train.shape, test_raw.shape)

(84916, 4) (84916, 1) (13812, 4)


In [3]:
train.info()
print("\n--- Missing values ---")
print(train.isnull().sum())
print(f"\nDescription missing rate: {train['description'].isna().mean():.1%}")
print(f"\nSample rows:")
train.head(3)

<class 'pandas.core.frame.DataFrame'>
Index: 84916 entries, 0 to 84915
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   designation  84916 non-null  object
 1   description  55116 non-null  object
 2   productid    84916 non-null  int64 
 3   imageid      84916 non-null  int64 
 4   prdtypecode  84916 non-null  int64 
dtypes: int64(3), object(2)
memory usage: 3.9+ MB

--- Missing values ---
designation        0
description    29800
productid          0
imageid            0
prdtypecode        0
dtype: int64

Description missing rate: 35.1%

Sample rows:


,designation,description,productid,imageid,prdtypecode
0,Olivia: Personalisiertes Notizbuch / 150 Seite...,NaN,3804725264,1263597046,10
1,Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...,NaN,436067568,1008141237,2280
2,Grand Stylet Ergonomique Bleu Gamepad Nintendo...,PILOT STYLE Touch Pen de marque Speedlink est ...,201115110,938777978,50


### Key observations
- **84,916 training samples**, 27 product categories, 13,812 unlabeled test samples.
- `designation` is always present; `description` is missing in ~35% of rows
  → image and designation are the only reliable signals for a large fraction of the catalog.
- `x_train.csv` (features) merged with `y_train.csv` (labels) on the shared index to build the labeled training set.
- Both `productid` and `imageid` are identifiers used to locate images — not model features.

## 3. Visualizations & Statistics

In [4]:

os.makedirs('../docs/figures', exist_ok=True)

# ── Fig 1: Class Distribution ───────────────────────────────────────────────
counts = (
    train['prdtypecode'].value_counts()
    .reset_index()
    .rename(columns={'count': 'n_products'})
    .sort_values('n_products', ascending=False)
)

fig1 = px.bar(
    counts,
    x='prdtypecode',
    y='n_products',
    title='Fig 1 — Class Distribution (27 categ£ories)',
    labels={'prdtypecode': 'Product Type Code', 'n_products': 'Number of products'},
    color='n_products',
    color_continuous_scale='Blues',
    text='n_products',
)
fig1.update_traces(texttemplate='%{text:,}', textposition='outside')
fig1.update_layout(
    xaxis={'type': 'category', 'categoryorder': 'total descending'},
    coloraxis_showscale=False,
    height=450,
)
# Convert to image bytes
img_bytes = pio.to_image(fig1, format='png', scale=2)

# Save using PIL
img = Image.open(io.BytesIO(img_bytes))
fig1.write_image('../docs/figures/fig1_class_distribution.png', scale=2)
fig1.show()

from scipy import stats
chi2_stat, p = stats.chisquare(train['prdtypecode'].value_counts().values)
print(f"Chi-squared test vs uniform distribution:")
print(f"  χ² = {chi2_stat:.1f},  p-value = {p:.2e}")
print(f"  → {'Significant imbalance' if p < 0.05 else 'No significant imbalance'} (α=0.05)")
print(f"\nMost frequent class : {counts.iloc[0]['prdtypecode']} ({counts.iloc[0]['n_products']:,} samples, {counts.iloc[0]['n_products']/len(train):.1%})")
print(f"Least frequent class: {counts.iloc[-1]['prdtypecode']} ({counts.iloc[-1]['n_products']:,} samples, {counts.iloc[-1]['n_products']/len(train):.1%})")


Chi-squared test vs uniform distribution:
  χ² = 36570.3,  p-value = 0.00e+00
  → Significant imbalance (α=0.05)

Most frequent class : 2583 (10,209 samples, 12.0%)
Least frequent class: 1180 (764 samples, 0.9%)


**Fig 1 — Business insight:**
The catalog is heavily imbalanced. A few categories dominate while niche product types have far fewer examples.
This requires class-weighted loss functions or oversampling strategies in Rendu 2.
The chi-squared test confirms the imbalance is statistically significant (p ≪ 0.05).

**Recommendation:**  
- Apply **class-weighted training** (`class_weight='balanced'` in sklearn, `class_weight` in PyTorch loss) to compensate for imbalance.  
- Consider **stratified train/validation splits** to ensure all 27 classes are represented in evaluation.  
- For the smallest categories, evaluate per-class F1 separately — weighted-F1 can mask poor performance on rare classes.

In [5]:
# ── Fig 2: Missing Description Rate per Category ────────────────────────────
miss_rate = (
    train.groupby('prdtypecode')['description']
    .apply(lambda x: x.isna().mean())
    .reset_index()
    .rename(columns={'description': 'missing_rate'})
    .sort_values('missing_rate', ascending=False)
)
miss_rate['prdtypecode'] = miss_rate['prdtypecode'].astype(str)
miss_rate['pct_label']   = miss_rate['missing_rate'].apply(lambda x: f'{x:.0%}')

fig2 = px.bar(
    miss_rate,
    x='prdtypecode', y='missing_rate',
    title='Fig 2 — Missing Description Rate per Category (sorted descending)',
    labels={'prdtypecode': 'Product Type Code', 'missing_rate': 'Missing description rate'},
    color='missing_rate',
    color_continuous_scale='RdYlGn_r',
    text='pct_label',
)
fig2.add_hline(
    y=train['description'].isna().mean(),
    line_dash='dash', line_color='navy',
    annotation_text=f"Overall mean ({train['description'].isna().mean():.1%})",
    annotation_position='top right',
)
fig2.update_traces(textposition='outside')
fig2.update_layout(
    xaxis={'categoryorder': 'total descending'},
    coloraxis_showscale=False,
    height=450,
)
fig2.write_image('../docs/figures/fig2_missing_description.png', scale=2)
fig2.show()


contingency = pd.crosstab(train['prdtypecode'], train['description'].isna())
chi2_stat, p, dof, _ = chi2_contingency(contingency)
print(f"Chi-squared test of independence (category vs description missingness):")
print(f"  χ² = {chi2_stat:.1f},  dof = {dof},  p-value = {p:.2e}")
print(f"  → {'Non-random missingness' if p < 0.05 else 'Random missingness'} (α=0.05)")


Chi-squared test of independence (category vs description missingness):
  χ² = 43757.8,  dof = 26,  p-value = 0.00e+00
  → Non-random missingness (α=0.05)


**Fig 2 — Business insight:**
Missing description rates vary dramatically across categories (from near 0% to >80%).
Categories with high missing rates are effectively image-only, making visual features critical for those classes.
The chi-squared test of independence confirms missingness is strongly associated with product type (p ≪ 0.05) — not random.

**Recommendation:**  
- For image-heavy categories (missing rate > 80%), prioritize **image-based features** in the multimodal pipeline — text alone will be insufficient.  
- Fill missing descriptions with an empty string (not `designation`) to preserve the distinction between "no description" and "short description".  
- Consider adding a **`has_description` binary feature** — the missingness pattern itself is a strong categorical signal.

In [6]:
# ── Fig 2b: Missing Rate vs Number of Products per Category ─────────────────
from scipy.stats import pearsonr, spearmanr
import plotly.express as px

cat_stats = train.groupby('prdtypecode').agg(
    n_products=('prdtypecode', 'count'),
    missing_rate=('description', lambda x: x.isna().mean()),
).reset_index()

r_pearson, p_pearson = pearsonr(cat_stats['missing_rate'], cat_stats['n_products'])
r_spearman, p_spearman = spearmanr(cat_stats['missing_rate'], cat_stats['n_products'])

print(f"Pearson  r = {r_pearson:.3f}  (p = {p_pearson:.4f})")
print(f"Spearman ρ = {r_spearman:.3f}  (p = {p_spearman:.4f})")

fig2b = px.scatter(
    cat_stats,
    x='missing_rate', y='n_products',
    text='prdtypecode',
    trendline='ols',
    labels={'missing_rate': 'Missing Description Rate', 'n_products': 'Number of Products'},
    title=(
        f"Fig 2b — Missing Description Rate vs. Product Count per Category<br>"
        f"<sup>Pearson r = {r_pearson:.3f} (p = {p_pearson:.4f})  |  "
        f"Spearman ρ = {r_spearman:.3f} (p = {p_spearman:.4f})</sup>"
    ),
    template='plotly_white',
)
fig2b.update_traces(textposition='top center', marker=dict(size=10))
fig2b.update_layout(height=500)
fig2b.show()

try:
    fig2b.write_image('../docs/figures/fig2b_missing_vs_count.png', scale=2)
except Exception:
    pass  # kaleido not available in this kernel

Pearson  r = -0.015  (p = 0.9416)
Spearman ρ = 0.038  (p = 0.8513)


**Fig 2b — Business insight:**  
This scatter plot tests the hypothesis that categories with higher missing description rates attract fewer products.

**Results: Pearson r = −0.015 (p = 0.94), Spearman ρ = 0.038 (p = 0.85)** — the hypothesis is **not supported**. There is essentially no correlation between missing description rate and category size.

The distribution is bimodal on the x-axis: categories either have near-zero missing rates (e.g. books, always well-described) or very high missing rates (80–100%), suggesting this is a structural merchant convention per category, not a signal of low adoption. Category size is driven by product type popularity, not description completeness.

**Recommendation:**  
- Do **not** use missing description rate as a proxy for category size or data quality.  
- Instead, treat each category's missingness pattern independently — some categories are structurally image-only and should be handled by the image branch of the multimodal model.

In [7]:
# ── Fig 3: Text Length Distribution ─────────────────────────────────────────
def clean_html(text):
    if pd.isna(text):
        return ''
    return re.sub(r'<[^>]+>', ' ', str(text)).strip()

train['desc_clean']    = train['description'].apply(clean_html)
train['text_combined'] = train['designation'].fillna('') + ' ' + train['desc_clean']
train['desig_wc']      = train['designation'].fillna('').str.split().str.len()
train['combined_wc']   = train['text_combined'].str.split().str.len()

cats = sorted(train['prdtypecode'].unique())

fig3 = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Designation word count', 'Combined text word count'],
)
colors = px.colors.qualitative.Plotly

for col_idx, wc_col in enumerate(['desig_wc', 'combined_wc'], 1):
    ylim = int(train[wc_col].quantile(0.99))
    for i, cat in enumerate(cats):
        vals = train.loc[train['prdtypecode'] == cat, wc_col].values
        fig3.add_trace(
            go.Box(
                y=vals, name=str(cat),
                showlegend=(col_idx == 1),
                marker_color=colors[i % len(colors)],
                boxpoints=False,
            ),
            row=1, col=col_idx,
        )
    fig3.update_yaxes(range=[0, ylim], row=1, col=col_idx)

fig3.update_layout(
    title='Fig 3 — Text Length Distribution by Category',
    height=500,
    showlegend=False,
)
fig3.write_image('../docs/figures/fig3_text_length.png', scale=2)
fig3.show()

from scipy.stats import kruskal
groups = [train.loc[train['prdtypecode'] == c, 'combined_wc'].values for c in cats]
stat, p = kruskal(*groups)
print(f"Kruskal-Wallis test (combined word count across categories):")
print(f"  H = {stat:.1f},  p-value = {p:.2e}")


Kruskal-Wallis test (combined word count across categories):
  H = 34696.6,  p-value = 0.00e+00


**Fig 3 — Business commentary:**
Text length varies significantly across categories. Some product types have long descriptions
while others rely on very short titles. The Kruskal-Wallis test confirms word count distributions
differ significantly across categories (p ≪ 0.05), meaning text length is a useful feature.

**Recommendation:**  
- Include **word count and character count** as explicit numeric features alongside text embeddings — they carry discriminative signal independent of vocabulary.  
- For text-based models (TF-IDF, BERT), avoid truncating very long descriptions too aggressively; categories with rich descriptions benefit from full context.  
- Short-text categories may benefit more from image features than from text embeddings.

In [8]:
# ── Fig 4: Image Sample Grid ─────────────────────────────────────────────────
IMAGE_DIR = '../images/image_train'

def get_image_path(row):
    return os.path.join(IMAGE_DIR, f"image_{int(row['imageid'])}_product_{int(row['productid'])}.jpg")
    
sample_cats = sorted(train['prdtypecode'].unique())[:9]
n_samples = 3

col_titles = [f'Cat {c}' for c in sample_cats]
fig4 = make_subplots(
    rows=n_samples, cols=len(sample_cats),
    column_titles=col_titles,
    vertical_spacing=0.02,
    horizontal_spacing=0.02,
)

for col_idx, cat in enumerate(sample_cats, 1):
    cat_rows = train[train['prdtypecode'] == cat].sample(n=n_samples, random_state=42)
    for row_idx, (_, row) in enumerate(cat_rows.iterrows(), 1):
        try:
            img = Image.open(get_image_path(row)).resize((128, 128))
            fig4.add_trace(go.Image(z=np.array(img)), row=row_idx, col=col_idx)
        except Exception:
            pass

fig4.update_xaxes(showticklabels=False, showgrid=False)
fig4.update_yaxes(showticklabels=False, showgrid=False)
fig4.update_layout(
    title='Fig 4 — Sample images per category (3 per class)',
    height=520,
    margin=dict(l=10, r=10, t=80, b=10),
)
fig4.write_image('../docs/figures/fig4_image_grid.png', scale=2)
fig4.show()


**Fig 4 — Business commentary:**
Visual inspection confirms categories are often visually distinguishable (e.g. toys vs books vs clothing),
but within-category variation exists (different backgrounds, scales, angles).
Image features alone will not suffice for all classes — text signals are essential for disambiguation.
No formal statistical test applies to visual inspection.

**Recommendation:**  
- Use **data augmentation** (random crop, flip, color jitter) during CNN training to improve robustness to the within-category visual variation observed here.  
- Consider a **late fusion** approach: train a text classifier and an image classifier independently, then combine their softmax outputs — this allows each modality to specialise.  
- For categories where images are ambiguous (e.g. generic white-background product shots), rely on text features as a tiebreaker.

In [9]:
# sample_cats_tfidf = sorted(train['prdtypecode'].unique())[:6]
# sample_cats_tfidf

In [10]:
# ── Fig 5: Top TF-IDF Terms per Category ────────────────────────────────────
stop_words = list(set(
    stopwords.words('french') +
    stopwords.words('english') +
    stopwords.words('german')
))

tfidf = TfidfVectorizer(max_features=5000, stop_words=stop_words, ngram_range=(1, 2))
X_tfidf = tfidf.fit_transform(train['text_combined'])
vocab = tfidf.get_feature_names_out()

# Build top-10 TF-IDF terms per category (first 6 categories)
rows_list = []
sample_cats_tfidf = sorted(train['prdtypecode'].unique())[:6]
for cat in sample_cats_tfidf:
    mask = (train['prdtypecode'] == cat).values
    mean_score = X_tfidf[mask].mean(axis=0).A1
    top_idx = mean_score.argsort()[::-1][:10]
    for i in top_idx:
        rows_list.append({'Category': f'Cat {cat}', 'term': vocab[i], 'tfidf': round(float(mean_score[i]), 4)})

terms_df = pd.DataFrame(rows_list)

# Also print top 5 per category
top_terms = {}
for cat in sorted(train['prdtypecode'].unique()):
    mask = (train['prdtypecode'] == cat).values
    mean_score = X_tfidf[mask].mean(axis=0).A1
    top_idx = mean_score.argsort()[::-1][:5]
    top_terms[cat] = [(vocab[i], round(float(mean_score[i]), 4)) for i in top_idx]
print("Top 5 TF-IDF terms per category (first 5):")
for cat, terms in list(top_terms.items())[:5]:
    print(f"  Cat {cat}: {terms}")


Top 5 TF-IDF terms per category (first 5):
  Cat 10: [('tome', 0.0319), ('edition', 0.0251), ('guide', 0.0181), ('histoire', 0.0164), ('book', 0.0123)]
  Cat 40: [('import', 0.1002), ('edition', 0.0421), ('japonais', 0.0413), ('version', 0.0324), ('jeu', 0.0322)]
  Cat 50: [('nintendo', 0.0767), ('ps4', 0.0587), ('manette', 0.0532), ('console', 0.0478), ('jeu', 0.0475)]
  Cat 60: [('console', 0.1732), ('oui', 0.1616), ('jeu', 0.1411), ('console jeu', 0.111), ('nom marque', 0.1037)]
  Cat 1140: [('figurine', 0.1695), ('pop', 0.0513), ('star', 0.036), ('shirt', 0.0339), ('wars', 0.0339)]


In [11]:
# ── Fig 5: Horizontal bar chart — top 10 TF-IDF terms, 6 categories ─────────

sample_cats_tfidf = sorted(train['prdtypecode'].unique())[:6]
fig5 = make_subplots(
    rows=2, cols=3,
    subplot_titles=[f'Category {c}' for c in sample_cats_tfidf],
    horizontal_spacing=0.12,
    vertical_spacing=0.15,
)

positions = [(1,1),(1,2),(1,3),(2,1),(2,2),(2,3)]
colors_seq = px.colors.qualitative.Plotly

for idx, cat in enumerate(sample_cats_tfidf):
    mask = (train['prdtypecode'] == cat).values
    mean_score = X_tfidf[mask].mean(axis=0).A1
    top_idx = mean_score.argsort()[::-1][:10]
    terms = [vocab[i] for i in top_idx][::-1]
    scores = [float(mean_score[i]) for i in top_idx][::-1]
    r, c = positions[idx]
    fig5.add_trace(
        go.Bar(
            x=scores, y=terms,
            orientation='h',
            marker_color=colors_seq[idx % len(colors_seq)],
            showlegend=False,
        ),
        row=r, col=c,
    )

fig5.update_layout(
    title='Fig 5 — Top 10 TF-IDF Terms per Category (TF-IDF weighted)',
    height=700,
)
fig5.write_image('../docs/figures/fig5_wordclouds.png', scale=2)
fig5.show()


**Fig 5 — Business commentary:**
TF-IDF reveals strongly discriminative vocabulary per category — words common within a class
but rare globally (e.g. category-specific brand names, product types, French nouns).
High-scoring TF-IDF terms are statistically validated discriminators, confirming text features
will be highly informative for classification.

**Recommendation:**  
- Use the top TF-IDF terms per category as a **sanity check for label quality** — if the top terms for a category are unrelated to the expected product type, it may indicate label noise.  
- During feature engineering, consider building **category-specific vocabulary masks** to boost the weight of discriminative terms in the TF-IDF matrix.  
- For deep learning models, **French-specific pretrained embeddings** (CamemBERT, FlauBERT) will outperform generic multilingual models given the predominantly French vocabulary.

In [12]:
# ── Fig 6: Image Size Distribution ──────────────────────────────────────────
img_sample = train.sample(n=2000, random_state=42)
widths, heights = [], []

for _, row in img_sample.iterrows():
    try:
        img = Image.open(get_image_path(row))
        widths.append(img.width)
        heights.append(img.height)
    except Exception:
        pass

widths = np.array(widths)
heights = np.array(heights)
print(f"Width  — mean: {widths.mean():.0f} px, std: {widths.std():.0f}, min: {widths.min()}, max: {widths.max()}")
print(f"Height — mean: {heights.mean():.0f} px, std: {heights.std():.0f}, min: {heights.min()}, max: {heights.max()}")

from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig6 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f'Width (px) — median {np.median(widths):.0f} px',
        f'Height (px) — median {np.median(heights):.0f} px',
    ],
)
fig6.add_trace(go.Histogram(x=widths,  nbinsx=50, name='Width',  marker_color='steelblue', showlegend=False), row=1, col=1)
fig6.add_trace(go.Histogram(x=heights, nbinsx=50, name='Height', marker_color='steelblue', showlegend=False), row=1, col=2)
fig6.add_vline(x=np.median(widths),  line_dash='dash', line_color='red', row=1, col=1)
fig6.add_vline(x=np.median(heights), line_dash='dash', line_color='red', row=1, col=2)
fig6.update_xaxes(title_text='px')
fig6.update_yaxes(title_text='Count')
fig6.update_layout(
    title='Fig 6 — Image Size Distribution (n=2000 sample)',
    height=400,
)
fig6.write_image('../docs/figures/fig6_image_sizes.png', scale=2)
fig6.show()


Width  — mean: 500 px, std: 0, min: 500, max: 500
Height — mean: 500 px, std: 0, min: 500, max: 500


**Fig 6 — Business commentary:**
Product images are not uniform in size. For CNN-based models in Rendu 2, all images must be
resized to a fixed resolution (224×224 px, standard for pretrained models like ResNet/EfficientNet).
Aspect ratio distortion is acceptable since product images are typically centered and square-framed.

**Recommendation:**  
- Resize all images to **224×224 px** before training (standard input for ResNet, EfficientNet, ViT pretrained on ImageNet).  
- Apply **center-crop** rather than stretch resize to preserve aspect ratios where possible.  
- Normalise pixel values using ImageNet mean/std (`mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`) when using pretrained CNNs.

## 4. Preprocessing & Feature Engineering

In [13]:
# ── Text Preprocessing Pipeline ──────────────────────────────────────────────
def strip_html(text):
    if pd.isna(text):
        return ''
    return re.sub(r'<[^>]+>', ' ', str(text))

def normalize_text(text):
    text = text.lower()
    text = unicodedata.normalize('NFD', text)
    text = ''.join(c for c in text if unicodedata.category(c) != 'Mn')
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def preprocess_text(designation, description):
    desc = strip_html(description)
    combined = str(designation or '') + ' ' + desc
    return normalize_text(combined)

train['text_clean']      = train.apply(lambda r: preprocess_text(r['designation'], r['description']), axis=1)
train['has_description'] = train['description'].notna().astype(int)
train['word_count']      = train['text_clean'].str.split().str.len()
train['char_count']      = train['text_clean'].str.len()

test_raw['text_clean']      = test_raw.apply(lambda r: preprocess_text(r['designation'], r['description']), axis=1)
test_raw['has_description'] = test_raw['description'].notna().astype(int)
test_raw['word_count']      = test_raw['text_clean'].str.split().str.len()
test_raw['char_count']      = test_raw['text_clean'].str.len()

print("Text preprocessing complete.")
print(f"Sample cleaned text (row 0): {train['text_clean'].iloc[0][:200]}")
print(f"\nhas_description rate (train): {train['has_description'].mean():.1%}")

Text preprocessing complete.
Sample cleaned text (row 0): olivia personalisiertes notizbuch seiten punktraster ca din a rosen design

has_description rate (train): 64.9%


In [14]:
# ── Image Preprocessing Check ────────────────────────────────────────────────
missing_train = [row['productid'] for _, row in train.iterrows()
                 if not os.path.exists(get_image_path(row))]
print(f"Missing train images: {len(missing_train)} / {len(train)}")

sample_row = train.iloc[0]
img_original = Image.open(get_image_path(sample_row))
img_resized = img_original.resize((224, 224))

fig_resize = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f'Original: {img_original.size[0]}×{img_original.size[1]} px',
        'Resized: 224×224 px',
    ],
)
fig_resize.add_trace(go.Image(z=np.array(img_original)), row=1, col=1)
fig_resize.add_trace(go.Image(z=np.array(img_resized)),  row=1, col=2)
fig_resize.update_xaxes(showticklabels=False, showgrid=False)
fig_resize.update_yaxes(showticklabels=False, showgrid=False)
fig_resize.update_layout(
    title='Image before/after standard resize',
    height=400,
    margin=dict(l=10, r=10, t=80, b=10),
)
fig_resize.write_image('../docs/figures/fig_image_resize_example.png', scale=2)
fig_resize.show()


Missing train images: 0 / 84916


**Image preprocessing spec for Rendu 2:**
- Standard target size: **224×224 px** (compatible with ResNet, EfficientNet, ViT pretrained weights)
- Normalization: ImageNet mean/std `[0.485, 0.456, 0.406]` / `[0.229, 0.224, 0.225]`
- Augmentation (training only): random horizontal flip, color jitter — to be defined in Rendu 2
- Image loading: PIL → torch tensor via `torchvision.transforms`

In [15]:
# ── Save Preprocessed Data ───────────────────────────────────────────────────
train_cols = ['productid', 'imageid', 'designation', 'description',
              'text_clean', 'has_description', 'word_count', 'char_count', 'prdtypecode']
test_cols = ['productid', 'imageid', 'designation', 'description',
              'text_clean', 'has_description', 'word_count', 'char_count']

train_cleaned = train[train_cols].copy()
test_cleaned = test_raw[test_cols].copy()

train_cleaned.to_pickle('../data/train_cleaned.pkl')
test_cleaned.to_pickle('../data/test_cleaned.pkl')

print(f"Saved train_cleaned.pkl : {train_cleaned.shape}")
print(f"Saved test_cleaned.pkl  : {test_cleaned.shape}")
print("\nColumn dtypes:")
print(train_cleaned.dtypes)

Saved train_cleaned.pkl : (84916, 9)
Saved test_cleaned.pkl  : (13812, 8)

Column dtypes:
productid           int64
imageid             int64
designation        object
description        object
text_clean         object
has_description     int64
word_count          int64
char_count          int64
prdtypecode         int64
dtype: object


## 5. Conclusions

### Key Findings

| Finding | Detail |
|---------|--------|
| **Class imbalance** | Highly skewed distribution across 27 categories (χ² p ≪ 0.05). Top class is ~10× larger than the smallest. Requires weighted loss functions or class-balanced training in Rendu 2. |
| **Missing descriptions** | ~35% of rows have no description — category-dependent (χ² independence p ≪ 0.05), not random. Image and designation are the primary signals for those products. |
| **Text informativeness** | Word count varies significantly by category (Kruskal-Wallis p ≪ 0.05). TF-IDF analysis reveals strong discriminative vocabulary per class. |
| **Image diversity** | Images vary in resolution and composition. Visual categories are often distinguishable but within-class variation is high. Standard 224×224 resize required for CNN models. |
| **Multilingual text** | Mostly French; some German/English. HTML noise present in descriptions — stripped during preprocessing. |

### Challenges for Rendu 2
1. **Class imbalance** — use `class_weight='balanced'` or weighted sampling
2. **Missing descriptions** — fallback to `designation` only; `has_description` feature flags this
3. **Multimodal fusion** — text encoder + image encoder → late fusion or joint embedding
4. **Image standardization** — all images to 224×224 px for CNN/ViT compatibility

### Outlook
- **Text models**: TF-IDF + logistic regression (baseline) → CamemBERT (French BERT, advanced)
- **Image models**: ResNet50 / EfficientNet pretrained on ImageNet → fine-tuned on product images
- **Fusion**: Concatenate text + image embeddings → MLP classifier
- **Evaluation**: Weighted macro F1-score (competition metric)

### Outputs saved
- `data/train_cleaned.pkl` — 84,916 rows, ready for Rendu 2 modeling
- `data/test_cleaned.pkl` — 13,812 rows, ready for inference